# 2.6 tf.data 基礎

這份 Notebook 示範如何從 NumPy array 建立 `tf.data.Dataset`，並使用 map、shuffle、batch、prefetch。

## 1. 載入套件與建立資料

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)
X, y = make_classification(n_samples=1200, n_features=10, n_informative=6, n_redundant=2, random_state=42)
X = X.astype('float32')
y = y.astype('float32')

## 2. 切分與標準化

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

## 3. 建立 Dataset

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

for features, label in train_ds.take(1):
    print(features.shape, label.numpy())

## 4. map、shuffle、batch、prefetch

In [ ]:
@tf.autograph.experimental.do_not_convert
def add_feature_noise(features, label):
    return features + tf.random.normal(tf.shape(features), stddev=0.01), label

train_ds = (train_ds
    .shuffle(buffer_size=len(x_train), seed=42)
    .map(add_feature_noise, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE))

test_ds = test_ds.batch(32).prefetch(tf.data.AUTOTUNE)

for batch_x, batch_y in train_ds.take(1):
    print(batch_x.shape, batch_y.shape)

## 5. 用 Dataset 訓練 Keras 模型

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, epochs=20, verbose=0)

pd.DataFrame([
    model.evaluate(train_ds, verbose=0, return_dict=True),
    model.evaluate(test_ds, verbose=0, return_dict=True),
], index=['train', 'test'])

## 6. 小結

`tf.data.Dataset` 可以把資料讀取與轉換流程整理成穩定管線，並直接提供給 `model.fit()`。